In [2]:
# IMPORTING LIBRARIES

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [3]:
# LOAD THE DATASET
df=pd.read_csv("../data/in-between/german-credit_risk_in-between_dataset.csv")
df.head()

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Credit Risk
0,67,male,2,own,NaN,little,1169,6,radio/TV,0
1,22,female,2,own,little,moderate,5951,48,radio/TV,1
2,49,male,1,own,little,NaN,2096,12,education,0
3,45,male,2,free,little,little,7882,42,furniture/equipment,0
4,53,male,2,free,little,little,4870,24,car,1


In [4]:
# shape of dataset
print("-"*50)
print(f"Shape of the Dataset : {df.shape}")
print("-"*50)

--------------------------------------------------
Shape of the Dataset : (954, 10)
--------------------------------------------------


In [5]:
# INFO & ISNULL COUNT
print("                      INFO")
print('='*50)
print(df.info())
print('='*50)
print("                 ISNULL COUNT")
print('='*50)
print()
print(df.isnull().sum())

                      INFO
<class 'pandas.DataFrame'>
RangeIndex: 954 entries, 0 to 953
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Age               954 non-null    int64
 1   Sex               954 non-null    str  
 2   Job               954 non-null    int64
 3   Housing           954 non-null    str  
 4   Saving accounts   779 non-null    str  
 5   Checking account  576 non-null    str  
 6   Credit amount     954 non-null    int64
 7   Duration          954 non-null    int64
 8   Purpose           954 non-null    str  
 9   Credit Risk       954 non-null    int64
dtypes: int64(5), str(5)
memory usage: 98.9 KB
None
                 ISNULL COUNT

Age                   0
Sex                   0
Job                   0
Housing               0
Saving accounts     175
Checking account    378
Credit amount         0
Duration              0
Purpose               0
Credit Risk           0
dtype: int64


In [6]:
# SPLITTING TARGET FEATURE AND INPUT FEATURES
TARGET='Credit Risk'

X=df.drop(columns=[TARGET])
y=df[TARGET]

# printing the value counts of the target feature
print(y.value_counts(dropna=False))

Credit Risk
0    665
1    289
Name: count, dtype: int64


In [7]:
# SPLIT THE DATASET INTO TRAIN AND TEST SETS
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [8]:
# SEPARATE NUMERICAL AND CATEGORICAL FEATURES
numeric_features=X_train.select_dtypes(include=np.number).columns.tolist()
categoric_features=X_train.select_dtypes(include=['str','object']).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categoric_features)

Numeric: ['Age', 'Job', 'Credit amount', 'Duration']
Categorical: ['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']


In [9]:
# CREATING PREPROCESSING PIPELINE FOR NUMERICAL AND CATEGORICAL FEATURES

# numeric pipeline
numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
# categoric pipeline
categoric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])
# combining both pipelines into a single preprocessor
preprocessor = ColumnTransformer(transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categoric_pipeline, categoric_features)
    ])

In [10]:
# FITTING THE PREPROCESSOR ON TRAINING DATA AND TRANSFORMING BOTH TRAIN AND TEST DATA
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# converting to dataset
X_train_processed = pd.DataFrame(X_train_processed, columns=preprocessor.get_feature_names_out())
X_test_processed = pd.DataFrame(X_test_processed, columns=preprocessor.get_feature_names_out())

# saving the dataframes to csv files
X_train_processed.to_csv("../data/processed/X_train_processed.csv", index=False)
X_test_processed.to_csv("../data/processed/X_test_processed.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

In [11]:
X_train_processed

,numeric__Age,numeric__Job,numeric__Credit amount,numeric__Duration,categorical__Sex_female,categorical__Sex_male,categorical__Housing_free,categorical__Housing_own,categorical__Housing_rent,categorical__Saving accounts_little,...,categorical__Checking account_moderate,categorical__Checking account_rich,categorical__Purpose_business,categorical__Purpose_car,categorical__Purpose_domestic appliances,categorical__Purpose_education,categorical__Purpose_furniture/equipment,categorical__Purpose_radio/TV,categorical__Purpose_repairs,categorical__Purpose_vacation/others
0,0.793959,-1.427676,1.020327,-0.219973,0.0,1.0,0.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1.615804,0.117454,-0.295809,0.278807,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,-0.941046,-1.427676,-0.329582,-0.469362,1.0,0.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,-0.210518,0.117454,-0.085157,-0.469362,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.976592,1.662584,-0.277703,0.029417,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
758,-1.032363,0.117454,1.144281,2.273924,0.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
759,-1.123679,0.117454,-0.983820,-1.217531,1.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
760,0.063431,1.662584,-0.316003,-0.718752,0.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
761,0.793959,0.117454,-0.734869,-0.718752,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
X_test_processed

,numeric__Age,numeric__Job,numeric__Credit amount,numeric__Duration,categorical__Sex_female,categorical__Sex_male,categorical__Housing_free,categorical__Housing_own,categorical__Housing_rent,categorical__Saving accounts_little,...,categorical__Checking account_moderate,categorical__Checking account_rich,categorical__Purpose_business,categorical__Purpose_car,categorical__Purpose_domestic appliances,categorical__Purpose_education,categorical__Purpose_furniture/equipment,categorical__Purpose_radio/TV,categorical__Purpose_repairs,categorical__Purpose_vacation/others
0,1.707120,0.117454,-0.354652,0.278807,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-0.667098,0.117454,-0.754019,-0.718752,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,1.250540,-1.427676,-0.700399,-0.718752,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,0.520011,-1.427676,-1.052761,-1.217531,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-0.849730,0.117454,-0.657224,-0.718752,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186,0.154747,0.117454,2.419331,2.273924,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
187,1.067908,1.662584,-0.718852,-1.217531,0.0,1.0,0.0,1.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
188,-0.484466,0.117454,-0.442742,-0.718752,0.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
189,1.067908,0.117454,0.105996,2.273924,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
